# Bổ sung PostgreSQL: work_mem và materialized view

Mặc định: 4 mức 1m, 5m, 10m và toàn bộ 20.342.188 dòng, mỗi mức 5 lần.
Baseline 4 MB, work_mem 32/64 MB: Q1–Q6; materialized view 4 MB: Q1 và Q5.
Tổng cộng 400 phép đo chính, thêm chi phí tạo/làm mới/dung lượng và EXPLAIN.

Chọn CPU → Run all → cho phép Drive chứa My Drive/book-sales-colab.
Mã nguồn nhúng sẵn; không cần tải script riêng. Notebook không sử dụng số liệu
cũ làm baseline và không thay đổi warehouse trên Drive. Giữ tab mở và theo dõi RAM.
Có thể sửa PLAN=['full'] ở ô đầu nếu chỉ muốn đo mức chính; baseline vẫn được đo lại.


In [ ]:
# 1. Kết nối Drive và kiểm tra môi trường
import json, os, pathlib, platform, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/book-sales-colab')
WORK = pathlib.Path('/content/book-sales-bigdata')
assert (DRIVE / 'manifest.json').exists(), f'Không thấy {DRIVE}. Hãy đặt thư mục book-sales-colab ngay trong My Drive.'
(DRIVE / 'results').mkdir(exist_ok=True)

ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
free_disk_gb = shutil.disk_usage('/content').free / 1e9
cpu = [l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')]
print('Python', platform.python_version(), '| RAM GB', round(ram_gb, 1), '| Ổ trống GB', round(free_disk_gb, 1), '| CPU', len(cpu), cpu[0])
assert free_disk_gb > 20, 'Ổ đĩa trống dưới 20 GB, hãy đổi runtime.'
from datetime import datetime, timezone
SESSION = datetime.now(timezone.utc).strftime('%m%d_%H%M%S')
SESSION_DIR = DRIVE / 'results' / ('pg_extended_' + SESSION)
SESSION_DIR.mkdir(exist_ok=False)
PLAN = ['1m', '5m', '10m', 'full']
REPEATS = 5
print('Results folder:', SESSION_DIR)


In [ ]:
# 2. Chép dữ liệu từ Drive về ổ của Colab và kiểm tra toàn vẹn
import hashlib
manifest = json.loads((DRIVE / 'manifest.json').read_text())
(WORK / 'data/warehouse').mkdir(parents=True, exist_ok=True)
(WORK / 'src').mkdir(exist_ok=True)
(WORK / 'results').mkdir(exist_ok=True)
for name, info in manifest.items():
    target = WORK / 'data/warehouse' / name
    shutil.copy(DRIVE / 'warehouse' / name, target)
    digest = hashlib.sha256()
    with open(target, 'rb') as handle:
        while chunk := handle.read(1 << 20):
            digest.update(chunk)
    assert target.stat().st_size == info['bytes'] and digest.hexdigest() == info['sha256'], f'{name} bị lỗi hoặc tải lên chưa xong.'
    print('OK', name, round(info['bytes'] / 1e6), 'MB')
(WORK / 'src/05_benchmark_baseline.py').write_text('"""Đo hệ thống mốc so sánh CHƯA TỐI ƯU trên 6 truy vấn Q1-Q6.\n\nHai hệ thống mốc:\n- pandas_csv: mỗi lần chạy đọc lại CSV, đủ mọi cột, kiểu dữ liệu mặc định.\n- postgres: cấu hình mặc định, không có khóa chính, index hay partition.\n\nCách chạy, từ thư mục gốc (server PostgreSQL phải đang chạy, xem pg_local.py):\n    .venv\\\\Scripts\\\\python src\\\\05_benchmark_baseline.py --sizes 1m 10m full --repeats 5\n\nCách đo:\n- Thứ tự truy vấn được xáo ngẫu nhiên trong mỗi vòng, có seed cố định.\n- PostgreSQL chạy một lần khởi động trước khi đo. Vì vậy số đo là trạng thái cache nóng.\n- PostgreSQL đo thời gian thực thi trên server, không tính truyền toàn bộ kết quả về client.\n- pandas được theo dõi RAM đỉnh và bị dừng nếu vượt giới hạn, để không ảnh hưởng ứng dụng khác.\n- Mỗi truy vấn tạo một "dấu kết quả" để đối chiếu giữa hai hệ thống.\n"""\nimport argparse\nimport csv\nimport json\nimport platform\nimport random\nimport shutil\nimport statistics\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent.parent\nWAREHOUSE = ROOT / "data" / "warehouse"\nBENCH = ROOT / "data" / "benchmark"\nSEED = 20261109\nSIZES = {"1m": 1_000_000, "5m": 5_000_000, "10m": 10_000_000, "full": None}\nQUERIES = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"]\nTABLES = ["dim_books", "dim_customers", "dim_date", "fact_sales"]\nPG_DSN = "host=127.0.0.1 port=55432 user=postgres dbname=book_sales"\n\nDDL = {\n    "dim_books": """book_key BIGINT, parent_asin TEXT, title TEXT, author TEXT, category TEXT, publisher TEXT,\n                    isbn_13 TEXT, publish_year INT, language TEXT, pages INT, list_price DOUBLE PRECISION,\n                    average_rating REAL, rating_number INT""",\n    "dim_customers": """customer_key BIGINT, user_id TEXT, first_purchase_date DATE, last_purchase_date DATE,\n                        total_purchases BIGINT""",\n    "dim_date": """date_key INT, date DATE, day INT, week INT, month INT, quarter INT, year INT, is_weekend BOOLEAN""",\n    "fact_sales": """sale_id BIGINT, date_key INT, customer_key BIGINT, book_key BIGINT, quantity SMALLINT,\n                     price_usd DOUBLE PRECISION, estimated_revenue DOUBLE PRECISION, rating REAL, helpful_vote INT""",\n}\n\n\ndef sql_for(query, customer):\n    return {\n        "Q1": """SELECT count(*), sum(r) FROM (\n                     SELECT d.year, d.month, sum(f.estimated_revenue) AS r\n                     FROM fact_sales f JOIN dim_date d USING (date_key) GROUP BY 1, 2) s""",\n        "Q2": """WITH c AS (SELECT book_key, count(*) AS n FROM fact_sales GROUP BY 1),\n                      r AS (SELECT c.book_key, c.n,\n                                   row_number() OVER (PARTITION BY b.category ORDER BY c.n DESC, c.book_key) AS rk\n                            FROM c JOIN dim_books b USING (book_key))\n                 SELECT count(*), sum(n), sum(book_key) FROM r WHERE rk <= 10""",\n        "Q3": f"""SELECT count(*), sum(sale_id) FROM (\n                      SELECT f.sale_id, b.title, d.date\n                      FROM fact_sales f JOIN dim_books b USING (book_key) JOIN dim_date d USING (date_key)\n                      WHERE f.customer_key = {int(customer)} ORDER BY f.sale_id) s""",\n        "Q4": """SELECT count(*), sum(n) FROM (\n                     SELECT d.year, count(DISTINCT f.customer_key) AS n\n                     FROM fact_sales f JOIN dim_date d USING (date_key) GROUP BY 1) s""",\n        "Q5": """WITH daily AS (SELECT date_key, sum(estimated_revenue) AS r FROM fact_sales GROUP BY 1),\n                      s AS (SELECT avg(coalesce(x.r, 0)) OVER (ORDER BY d.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS a\n                            FROM dim_date d LEFT JOIN daily x USING (date_key))\n                 SELECT count(*), sum(a) FROM s""",\n        "Q6": """WITH m AS (SELECT f.customer_key, d.date, f.estimated_revenue\n                            FROM fact_sales f JOIN dim_date d USING (date_key)),\n                      g AS (SELECT customer_key, max(date) AS last_date, count(*) AS frequency,\n                                   coalesce(sum(estimated_revenue), 0) AS monetary\n                            FROM m GROUP BY 1)\n                 SELECT count(*), sum(frequency), sum(monetary),\n                        sum((SELECT max(date) FROM m) - last_date) FROM g""",\n    }[query]\n\n\ndef run_pandas(query, folder, customer):\n    """Cách làm thông thường, chưa tối ưu: đọc lại CSV đầy đủ mỗi lần chạy."""\n    import pandas as pd\n\n    def load(name, **options):\n        return pd.read_csv(folder / f"{name}.csv", **options)\n\n    fact = load("fact_sales")\n    if query == "Q1":\n        merged = fact.merge(load("dim_date"), on="date_key")\n        result = merged.groupby(["year", "month"])["estimated_revenue"].sum()\n        return [len(result), float(result.sum())]\n    if query == "Q2":\n        counts = fact.groupby("book_key").size().rename("n").reset_index()\n        merged = counts.merge(load("dim_books", dtype={"isbn_13": str}), on="book_key")\n        merged = merged.sort_values(["category", "n", "book_key"], ascending=[True, False, True])\n        top = merged.groupby("category", dropna=False).head(10)\n        return [len(top), int(top["n"].sum()), int(top["book_key"].sum())]\n    if query == "Q3":\n        rows = fact[fact["customer_key"] == customer]\n        rows = rows.merge(load("dim_books", dtype={"isbn_13": str}), on="book_key")\n        rows = rows.merge(load("dim_date"), on="date_key").sort_values("sale_id")\n        return [len(rows), int(rows["sale_id"].sum())]\n    if query == "Q4":\n        merged = fact.merge(load("dim_date"), on="date_key")\n        result = merged.groupby("year")["customer_key"].nunique()\n        return [len(result), int(result.sum())]\n    dates = load("dim_date", parse_dates=["date"])\n    merged = fact.merge(dates, on="date_key")\n    if query == "Q5":\n        daily = merged.groupby("date")["estimated_revenue"].sum()\n        daily = daily.reindex(dates.sort_values("date")["date"], fill_value=0)\n        rolling = daily.rolling(7, min_periods=1).mean()\n        return [len(rolling), float(rolling.sum())]\n    if query == "Q6":\n        grouped = merged.groupby("customer_key").agg(\n            last_date=("date", "max"), frequency=("sale_id", "count"), monetary=("estimated_revenue", "sum"))\n        recency = (merged["date"].max() - grouped["last_date"]).dt.days\n        return [len(grouped), int(grouped["frequency"].sum()), float(grouped["monetary"].sum()), int(recency.sum())]\n    raise ValueError(query)\n\n\ndef worker(query, folder, customer):\n    sys.stdout.reconfigure(encoding="utf-8")\n    started = time.perf_counter()\n    fingerprint = run_pandas(query, Path(folder), int(customer))\n    print(json.dumps({"seconds": time.perf_counter() - started, "fingerprint": fingerprint}))\n\n\ndef prepare(label):\n    """Xuất CSV cho một quy mô. Bảng chiều chỉ giữ các dòng được bảng sự kiện dùng tới."""\n    import duckdb\n\n    target = BENCH / label\n    if (target / "params.json").exists():\n        return target\n    temp = BENCH / (label + ".tmp")\n    shutil.rmtree(temp, ignore_errors=True)\n    temp.mkdir(parents=True)\n    wh = WAREHOUSE.as_posix()\n    out = temp.as_posix()\n    con = duckdb.connect()\n    con.execute("SET memory_limit=\'4GB\'")\n    con.execute(f"SET temp_directory=\'{(BENCH / \'spill\').as_posix()}\'")\n    limit = SIZES[label]\n    if limit is None:\n        con.execute(f"CREATE VIEW fact AS SELECT * FROM \'{wh}/fact_sales.parquet\'")\n    else:\n        con.execute(f"""CREATE TABLE fact AS SELECT * FROM \'{wh}/fact_sales.parquet\'\n                        ORDER BY hash(sale_id, {SEED}), sale_id LIMIT {limit}""")\n    copies = {\n        "fact_sales": "SELECT * FROM fact",\n        "dim_date": f"SELECT * FROM \'{wh}/dim_date.parquet\'",\n        "dim_books": f"SELECT * FROM \'{wh}/dim_books.parquet\' b SEMI JOIN (SELECT DISTINCT book_key FROM fact) f USING (book_key)",\n        "dim_customers": f"SELECT * FROM \'{wh}/dim_customers.parquet\' c SEMI JOIN (SELECT DISTINCT customer_key FROM fact) f USING (customer_key)",\n    }\n    for table, query in copies.items():\n        con.execute(f"COPY ({query}) TO \'{out}/{table}.csv\' (FORMAT csv, HEADER)")\n    customer, purchases = con.execute(\n        "SELECT customer_key, count(*) n FROM fact GROUP BY 1 ORDER BY n DESC, customer_key LIMIT 1").fetchone()\n    rows = {t: con.execute(f"SELECT count(*) FROM read_csv(\'{out}/{t}.csv\', all_varchar=true, quote=\'\\"\')").fetchone()[0]\n            for t in TABLES}\n    con.close()\n    params = {"customer_key": int(customer), "customer_purchases": int(purchases), "rows": rows,\n              "csv_mb": {t: round((temp / f"{t}.csv").stat().st_size / 1e6, 1) for t in TABLES}}\n    (temp / "params.json").write_text(json.dumps(params, indent=2), encoding="utf-8")\n    if target.exists():\n        shutil.rmtree(target)\n    temp.rename(target)\n    return target\n\n\ndef load_postgres(label, folder, params):\n    import psycopg\n\n    schema = f"s_{label}"\n    with psycopg.connect(PG_DSN, autocommit=True) as con:\n        done = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_schema=%s", (schema,)).fetchone()[0]\n        if done == len(TABLES):\n            info_path = folder / "postgres_load.json"\n            if info_path.exists():\n                return json.loads(info_path.read_text(encoding="utf-8"))\n        con.execute(f"DROP SCHEMA IF EXISTS {schema} CASCADE")\n        con.execute(f"CREATE SCHEMA {schema}")\n        seconds = {}\n        for table in TABLES:\n            con.execute(f"CREATE TABLE {schema}.{table} ({DDL[table]})")\n            started = time.perf_counter()\n            with con.cursor() as cur, open(folder / f"{table}.csv", "rb") as source:\n                with cur.copy(f"COPY {schema}.{table} FROM STDIN (FORMAT csv, HEADER true, NULL \'\')") as copy:\n                    while chunk := source.read(1024 * 1024):\n                        copy.write(chunk)\n            seconds[table] = round(time.perf_counter() - started, 3)\n            count = con.execute(f"SELECT count(*) FROM {schema}.{table}").fetchone()[0]\n            if count != params["rows"][table]:\n                sys.exit(f"Nạp thiếu {table}: {count} so với {params[\'rows\'][table]}")\n            con.execute(f"ANALYZE {schema}.{table}")\n        size = {t: con.execute("SELECT pg_total_relation_size(%s)", (f"{schema}.{t}",)).fetchone()[0] for t in TABLES}\n    info = {"copy_seconds": seconds, "table_mb": {t: round(v / 1e6, 1) for t, v in size.items()}}\n    (folder / "postgres_load.json").write_text(json.dumps(info, indent=2), encoding="utf-8")\n    return info\n\n\ndef run_postgres(label, query, customer, timeout):\n    import psycopg\n\n    with psycopg.connect(PG_DSN, autocommit=True) as con:\n        con.execute(f"SET search_path TO s_{label}")\n        con.execute(f"SET statement_timeout = {int(timeout * 1000)}")\n        started = time.perf_counter()\n        try:\n            row = con.execute(sql_for(query, customer)).fetchone()\n        except psycopg.errors.QueryCanceled:\n            return {"status": "timeout", "seconds": time.perf_counter() - started}\n        seconds = time.perf_counter() - started\n    return {"status": "ok", "seconds": seconds, "fingerprint": [float(v) if v is not None else None for v in row]}\n\n\ndef run_pandas_guarded(query, folder, customer, args):\n    """Chạy pandas trong tiến trình con, dừng nếu vượt RAM, thời gian hoặc máy sắp hết bộ nhớ."""\n    import psutil\n\n    process = subprocess.Popen(\n        [sys.executable, str(Path(__file__).resolve()), "--worker", query, str(folder), str(customer)],\n        stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding="utf-8",\n    )\n    handle = psutil.Process(process.pid)\n\n    def family():\n        # Python trong venv trên Windows là tiến trình trung gian, tiến trình thật là tiến trình con của nó.\n        try:\n            return [handle] + handle.children(recursive=True)\n        except psutil.Error:\n            return []\n\n    def total_rss(processes):\n        total = 0\n        for item in processes:\n            try:\n                total += item.memory_info().rss\n            except psutil.Error:\n                pass\n        return total\n\n    started = time.perf_counter()\n    peak, status = 0, "ok"\n    members = []\n    while process.poll() is None:\n        members = family() or members\n        peak = max(peak, total_rss(members))\n        if peak > args.max_rss_gb * 1e9:\n            status = "exceeded_ram_limit"\n        elif psutil.virtual_memory().available < args.min_free_gb * 1e9:\n            status = "stopped_low_system_memory"\n        elif time.perf_counter() - started > args.timeout:\n            status = "timeout"\n        if status != "ok":\n            for item in reversed(members):\n                try:\n                    item.kill()\n                except psutil.Error:\n                    pass\n            process.kill()\n            break\n        time.sleep(0.05)\n    out, err = process.communicate()\n    result = {"status": status, "peak_gb": round(peak / 1e9, 3), "seconds": time.perf_counter() - started}\n    if status == "ok":\n        if process.returncode != 0:\n            result["status"] = "error"\n            result["error"] = err.strip()[-600:]\n        else:\n            result.update(json.loads(out.strip().splitlines()[-1]))\n    return result\n\n\ndef same(left, right):\n    if left is None or right is None or len(left) != len(right):\n        return False\n    return all(abs(float(a) - float(b)) <= 1e-6 * max(1.0, abs(float(a))) for a, b in zip(left, right))\n\n\ndef machine_info():\n    import psutil\n\n    return {\n        "cpu": platform.processor(), "logical_cpus": psutil.cpu_count(),\n        "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),\n        "available_ram_gb_at_start": round(psutil.virtual_memory().available / 1e9, 1),\n        "cpu_percent_at_start": psutil.cpu_percent(interval=1),\n        "python": platform.python_version(), "os": platform.platform(),\n    }\n\n\ndef main():\n    sys.stdout.reconfigure(encoding="utf-8")\n    sys.stderr.reconfigure(encoding="utf-8")\n    if len(sys.argv) > 1 and sys.argv[1] == "--worker":\n        worker(*sys.argv[2:5])\n        return\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--sizes", nargs="+", choices=SIZES, default=["1m"])\n    parser.add_argument("--engines", nargs="+", choices=["pandas_csv", "postgres"], default=["pandas_csv", "postgres"])\n    parser.add_argument("--repeats", type=int, default=5)\n    parser.add_argument("--tag", default="baseline")\n    parser.add_argument("--max-rss-gb", type=float, default=10)\n    parser.add_argument("--min-free-gb", type=float, default=1.5)\n    parser.add_argument("--timeout", type=float, default=900)\n    args = parser.parse_args()\n\n    out_dir = ROOT / "results" / args.tag\n    out_dir.mkdir(parents=True, exist_ok=True)\n    summary = {"machine": machine_info(), "sizes": {}}\n    rng = random.Random(SEED)\n    with open(out_dir / "runs.csv", "w", newline="", encoding="utf-8") as raw:\n        writer = csv.writer(raw)\n        writer.writerow(["size", "engine", "query", "repeat", "status", "seconds", "peak_gb", "fingerprint"])\n        for label in args.sizes:\n            folder = prepare(label)\n            params = json.loads((folder / "params.json").read_text(encoding="utf-8"))\n            customer = params["customer_key"]\n            info = {"params": params}\n            if "postgres" in args.engines:\n                info["postgres_load"] = load_postgres(label, folder, params)\n                for query in QUERIES:  # lượt khởi động, không tính vào kết quả\n                    run_postgres(label, query, customer, args.timeout)\n            stopped, prints, seconds, statuses, peaks = set(), {}, {}, {}, {}\n            for repeat in range(1, args.repeats + 1):\n                plan = [(e, q) for e in args.engines for q in QUERIES]\n                rng.shuffle(plan)\n                for engine, query in plan:\n                    key = (engine, query)\n                    if key in stopped:\n                        continue\n                    if engine == "postgres":\n                        result = run_postgres(label, query, customer, args.timeout)\n                    else:\n                        result = run_pandas_guarded(query, folder, customer, args)\n                    status = result["status"]\n                    statuses[key] = status\n                    writer.writerow([label, engine, query, repeat, status, round(result["seconds"], 4),\n                                     result.get("peak_gb", ""), json.dumps(result.get("fingerprint"))])\n                    raw.flush()\n                    print(label, engine, query, repeat, status, round(result["seconds"], 2), result.get("peak_gb", ""),\n                          result.get("error", ""), flush=True)\n                    if status != "ok":\n                        stopped.add(key)\n                        continue\n                    seconds.setdefault(key, []).append(result["seconds"])\n                    prints.setdefault(key, result["fingerprint"])\n                    if "peak_gb" in result:\n                        peaks[key] = max(peaks.get(key, 0), result["peak_gb"])\n            table = {}\n            for engine, query in [(e, q) for e in args.engines for q in QUERIES]:\n                key = (engine, query)\n                values = seconds.get(key, [])\n                table[f"{engine}/{query}"] = {\n                    "status": statuses.get(key), "runs": len(values),\n                    "median_s": round(statistics.median(values), 4) if values else None,\n                    "min_s": round(min(values), 4) if values else None,\n                    "max_s": round(max(values), 4) if values else None,\n                    "peak_gb": peaks.get(key),\n                }\n            info["results"] = table\n            if len(args.engines) == 2:\n                info["fingerprints_match"] = {\n                    q: same(prints.get(("pandas_csv", q)), prints.get(("postgres", q)))\n                    if ("pandas_csv", q) in prints and ("postgres", q) in prints else None\n                    for q in QUERIES\n                }\n            summary["sizes"][label] = info\n            (out_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps({k: v.get("fingerprints_match") for k, v in summary["sizes"].items()}, indent=2))\n    print(f"Xong -> {out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK / 'src/06_benchmark_optimized.py').write_text('"""Matched end-to-end benchmarks. See BENCHMARK_OPTIMIZED.md."""\nimport argparse\nimport csv\nimport importlib.util\nimport importlib.metadata\nimport json\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nspec = importlib.util.spec_from_file_location(\'baseline\', ROOT / \'src/05_benchmark_baseline.py\')\nbase = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(base)\nENGINES = [\'pandas_csv\', \'pandas_columns\', \'pandas_parquet\', \'duckdb\', \'postgres\', \'postgres_index\']\nCOLS = {\n    \'Q1\': {\'fact_sales\': [\'date_key\', \'estimated_revenue\'], \'dim_date\': [\'date_key\', \'year\', \'month\']},\n    \'Q2\': {\'fact_sales\': [\'book_key\'], \'dim_books\': [\'book_key\', \'category\']},\n    \'Q3\': {\'fact_sales\': [\'customer_key\', \'sale_id\', \'book_key\', \'date_key\'], \'dim_books\': [\'book_key\', \'title\'], \'dim_date\': [\'date_key\', \'date\']},\n    \'Q4\': {\'fact_sales\': [\'date_key\', \'customer_key\'], \'dim_date\': [\'date_key\', \'year\']},\n    \'Q5\': {\'fact_sales\': [\'date_key\', \'estimated_revenue\'], \'dim_date\': [\'date_key\', \'date\']},\n    \'Q6\': {\'fact_sales\': [\'customer_key\', \'date_key\', \'estimated_revenue\'], \'dim_date\': [\'date_key\', \'date\']},\n}\n\n\ndef sql(q, customer):\n    return {\n        \'Q1\': \'SELECT d.year,d.month,sum(f.estimated_revenue) AS revenue FROM fact_sales f JOIN dim_date d USING(date_key) GROUP BY 1,2 ORDER BY 1,2\',\n        \'Q2\': \'\'\'WITH c AS (SELECT book_key,count(*) AS n FROM fact_sales GROUP BY 1),\n                   r AS (SELECT b.category,c.book_key,c.n,row_number() OVER (PARTITION BY b.category ORDER BY c.n DESC,c.book_key) AS rk FROM c JOIN dim_books b USING(book_key))\n                 SELECT category,book_key,n FROM r WHERE rk<=10 ORDER BY category NULLS LAST,n DESC,book_key\'\'\',\n        \'Q3\': f\'\'\'SELECT f.sale_id,b.title,d.date FROM fact_sales f JOIN dim_books b USING(book_key)\n                  JOIN dim_date d USING(date_key) WHERE f.customer_key={int(customer)} ORDER BY f.sale_id\'\'\',\n        \'Q4\': \'SELECT d.year,count(DISTINCT f.customer_key) AS n FROM fact_sales f JOIN dim_date d USING(date_key) GROUP BY 1 ORDER BY 1\',\n        \'Q5\': \'\'\'WITH daily AS (SELECT date_key,sum(estimated_revenue) AS revenue FROM fact_sales GROUP BY 1)\n                 SELECT d.date,avg(coalesce(x.revenue,0)) OVER (ORDER BY d.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling\n                 FROM dim_date d LEFT JOIN daily x USING(date_key) ORDER BY d.date\'\'\',\n        \'Q6\': \'\'\'WITH m AS (SELECT f.customer_key,d.date,f.estimated_revenue FROM fact_sales f JOIN dim_date d USING(date_key))\n                 SELECT customer_key,CAST((SELECT max(date) FROM m)-max(date) AS INTEGER) AS recency,\n                 count(*) AS frequency,coalesce(sum(estimated_revenue),0) AS monetary FROM m GROUP BY 1 ORDER BY 1\'\'\',\n    }[q]\n\n\ndef pandas_query(engine, q, folder, customer):\n    import pandas as pd\n    def load(table):\n        columns = COLS[q][table]\n        if engine == \'pandas_parquet\':\n            return pd.read_parquet(folder / f\'{table}.parquet\', columns=columns)\n        options = {\'usecols\': columns} if engine == \'pandas_columns\' else {}\n        if table == \'dim_books\' and engine == \'pandas_csv\':\n            options[\'dtype\'] = {\'isbn_13\': str}\n        return pd.read_csv(folder / f\'{table}.csv\', **options)\n    f = load(\'fact_sales\')\n    if q == \'Q2\':\n        c = f.groupby(\'book_key\').size().rename(\'n\').reset_index()\n        r = c.merge(load(\'dim_books\'), on=\'book_key\')\n        return r.sort_values([\'category\', \'n\', \'book_key\'], ascending=[True, False, True]).groupby(\'category\', dropna=False).head(10)[[\'category\', \'book_key\', \'n\']].reset_index(drop=True)\n    if q == \'Q3\':\n        r = f[f.customer_key == customer].merge(load(\'dim_books\'), on=\'book_key\').merge(load(\'dim_date\'), on=\'date_key\')\n        r[\'date\'] = pd.to_datetime(r[\'date\'])\n        return r.sort_values(\'sale_id\')[[\'sale_id\', \'title\', \'date\']].reset_index(drop=True)\n    d = load(\'dim_date\')\n    if q in [\'Q5\', \'Q6\']:\n        d[\'date\'] = pd.to_datetime(d[\'date\'])\n    r = f.merge(d, on=\'date_key\')\n    if q == \'Q1\':\n        return r.groupby([\'year\', \'month\']).estimated_revenue.sum(min_count=1).rename(\'revenue\').reset_index()\n    if q == \'Q4\':\n        return r.groupby(\'year\').customer_key.nunique().rename(\'n\').reset_index()\n    if q == \'Q5\':\n        daily = r.groupby(\'date\').estimated_revenue.sum(min_count=1)\n        values = daily.reindex(d.sort_values(\'date\')[\'date\']).fillna(0).rolling(7, min_periods=1).mean()\n        return values.rename(\'rolling\').rename_axis(\'date\').reset_index()\n    g = r.groupby(\'customer_key\').agg(last_date=(\'date\', \'max\'), frequency=(\'customer_key\', \'size\'), monetary=(\'estimated_revenue\', \'sum\'))\n    g[\'recency\'] = (r.date.max() - g.last_date).dt.days\n    return g.reset_index()[[\'customer_key\', \'recency\', \'frequency\', \'monetary\']]\n\n\ndef worker(engine, q, folder, customer, output, schema, timeout):\n    import pandas as pd\n    folder = Path(folder)\n    started = time.perf_counter()\n    if engine.startswith(\'pandas\'):\n        result = pandas_query(engine, q, folder, int(customer))\n    elif engine == \'duckdb\':\n        import duckdb\n        with duckdb.connect() as con:\n            con.execute("SET memory_limit=\'2GB\'")\n            con.execute(\'SET threads=2\')\n            for table in COLS[q]:\n                con.execute(f"CREATE VIEW {table} AS SELECT * FROM read_parquet(\'{folder.as_posix()}/{table}.parquet\')")\n            result = con.execute(sql(q, customer)).fetchdf()\n    else:\n        import psycopg\n        with psycopg.connect(base.PG_DSN, autocommit=True) as con:\n            con.execute(f\'SET search_path TO {schema}\')\n            con.execute(f\'SET statement_timeout={int(float(timeout)*1000)}\')\n            with con.cursor() as cur:\n                cur.execute(sql(q, customer))\n                result = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])\n    elapsed = time.perf_counter() - started\n    # Persist outside the timed region; compare actual rows, not aggregate fingerprints.\n    result.to_parquet(output, index=False)\n    print(json.dumps({\'seconds\': elapsed, \'rows\': len(result)}))\n\n\ndef guarded(engine, q, folder, customer, output, schema, args):\n    import psutil\n    command = [sys.executable, str(Path(__file__).resolve()), \'--worker\', engine, q, str(folder), str(customer), str(output), schema, str(args.timeout)]\n    p = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding=\'utf-8\')\n    handle = psutil.Process(p.pid)\n    start, peak, status = time.perf_counter(), 0, \'ok\'\n    while p.poll() is None:\n        try:\n            members = [handle] + handle.children(recursive=True)\n            rss = sum(x.memory_info().rss for x in members if x.is_running())\n            peak = max(peak, rss)\n            if rss > args.max_rss_gb * 1e9 or psutil.virtual_memory().available < args.min_free_gb * 1e9:\n                status = \'memory_limit\'\n            elif time.perf_counter() - start > args.timeout:\n                status = \'timeout\'\n            if status != \'ok\':\n                for member in reversed(members):\n                    try:\n                        member.kill()\n                    except psutil.Error:\n                        pass\n                break\n        except psutil.Error:\n            pass\n        time.sleep(.05)\n    out, err = p.communicate()\n    r = {\'status\': status, \'client_peak_gb\': peak / 1e9}\n    if status == \'ok\':\n        if p.returncode:\n            r.update(status=\'error\', error=err[-2000:])\n        else:\n            r.update(json.loads(out.strip().splitlines()[-1]))\n    return r\n\n\ndef compare(reference, actual):\n    import pandas as pd\n    from pandas.api.types import is_numeric_dtype\n    a, b = pd.read_parquet(reference), pd.read_parquet(actual)\n    if list(a.columns) != list(b.columns) or a.shape != b.shape:\n        raise AssertionError(f\'Shape/columns differ: {a.shape} / {b.shape}\')\n    # SQL collation depends on the server locale. Compare rows in a common\n    # client-side order, outside the timed region; do not discard duplicates.\n    if \'date\' in a.columns:\n        a[\'date\'] = pd.to_datetime(a[\'date\'])\n        b[\'date\'] = pd.to_datetime(b[\'date\'])\n    keys = list(a.columns)\n    a = a.sort_values(keys, kind=\'stable\', na_position=\'last\').reset_index(drop=True)\n    b = b.sort_values(keys, kind=\'stable\', na_position=\'last\').reset_index(drop=True)\n    for col in a:\n        if is_numeric_dtype(a[col]) and is_numeric_dtype(b[col]):\n            # Integers must match exactly; tolerance only for floating point aggregation.\n            floating = a[col].dtype.kind == \'f\' or b[col].dtype.kind == \'f\'\n            pd.testing.assert_series_equal(a[col], b[col], check_dtype=False, check_exact=not floating, rtol=1e-8, atol=1e-6)\n        elif \'date\' == col:\n            pd.testing.assert_series_equal(pd.to_datetime(a[col]), pd.to_datetime(b[col]), check_dtype=False)\n        else:\n            pd.testing.assert_series_equal(a[col].astype(\'string\'), b[col].astype(\'string\'))\n\n\ndef prepare_parquet(folder):\n    import duckdb\n    records = {}\n    with duckdb.connect() as con:\n        for table in base.TABLES:\n            source, target = folder / f\'{table}.csv\', folder / f\'{table}.parquet\'\n            started = time.perf_counter()\n            if not target.exists():\n                # Explicit schema preserves CSV strings, including ISBN and missing categories.\n                columns = \',\'.join(f"\'{part.strip().split()[0]}\':\'{part.strip().split()[1]}\'" for part in base.DDL[table].split(\',\'))\n                con.execute(f"COPY (SELECT * FROM read_csv(\'{source.as_posix()}\', header=true, columns={{{columns}}})) TO \'{target.as_posix()}\' (FORMAT parquet, COMPRESSION zstd)")\n                created = True\n            else:\n                created = False\n            records[table] = {\'created\': created, \'prepare_seconds\': time.perf_counter()-started, \'csv_bytes\': source.stat().st_size, \'parquet_bytes\': target.stat().st_size}\n    return records\n\n\ndef setup_postgres(label):\n    import psycopg\n    schema = f\'opt_{label}\'\n    with psycopg.connect(base.PG_DSN, autocommit=True) as con:\n        if con.execute(\'SELECT 1 FROM information_schema.schemata WHERE schema_name=%s\', (schema,)).fetchone():\n            raise RuntimeError(f\'{schema} already exists; use a fresh --tag (schema is derived from tag).\')\n        con.execute(f\'CREATE SCHEMA {schema}\')\n        started = time.perf_counter()\n        con.execute(f\'CREATE TABLE {schema}.fact_sales AS TABLE s_{label.split("_", 1)[0]}.fact_sales\')\n        for table in [\'dim_books\', \'dim_date\', \'dim_customers\']:\n            con.execute(f\'CREATE VIEW {schema}.{table} AS SELECT * FROM s_{label.split("_", 1)[0]}.{table}\')\n        con.execute(f\'ANALYZE {schema}.fact_sales\')\n        copy_seconds = time.perf_counter()-started\n        started = time.perf_counter()\n        con.execute(f\'CREATE INDEX ON {schema}.fact_sales(customer_key)\')\n        con.execute(f\'ANALYZE {schema}.fact_sales\')\n        info = {\'schema\': schema, \'copy_seconds\': copy_seconds, \'index_build_seconds\': time.perf_counter()-started,\n                \'index_bytes\': con.execute(\'SELECT pg_indexes_size(%s)\', (f\'{schema}.fact_sales\',)).fetchone()[0]}\n    return schema, info\n\n\ndef main():\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    if len(sys.argv)>1 and sys.argv[1]==\'--worker\':\n        worker(*sys.argv[2:]); return\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--sizes\', nargs=\'+\', choices=base.SIZES, default=[\'1m\'])\n    p.add_argument(\'--engines\', nargs=\'+\', choices=ENGINES, default=ENGINES[:4])\n    p.add_argument(\'--queries\', nargs=\'+\', choices=base.QUERIES, default=base.QUERIES)\n    p.add_argument(\'--repeats\', type=int, default=5)\n    p.add_argument(\'--tag\', required=True)\n    p.add_argument(\'--timeout\', type=float, default=600)\n    p.add_argument(\'--max-rss-gb\', type=float, default=6)\n    p.add_argument(\'--min-free-gb\', type=float, default=1.5)\n    args=p.parse_args()\n    if args.repeats < 1 or not args.tag.replace(\'_\',\'\').isalnum() or len(args.tag)>32:\n        p.error(\'repeats must be positive; tag must be <=32 alphanumeric/underscore characters\')\n    out=ROOT/\'results\'/args.tag\n    out.mkdir(parents=True, exist_ok=False)\n    versions={name:importlib.metadata.version(name) for name in [\'pandas\',\'duckdb\',\'pyarrow\',\'psutil\']}\n    summary={\'machine\':base.machine_info(), \'versions\':versions, \'arguments\':vars(args), \'timing\':\'Read/query + materialize full output in client; imports and result serialization excluded. RSS includes serialization; PostgreSQL server RAM excluded. OS cache uncontrolled.\', \'sizes\':{}}\n    rng=random.Random(base.SEED)\n    failed=False\n    with (out/\'runs.jsonl\').open(\'w\',encoding=\'utf-8\') as log:\n        for size in args.sizes:\n            folder=base.prepare(size)\n            params=json.loads((folder/\'params.json\').read_text())\n            item={\'parquet\':prepare_parquet(folder), \'runs\':[]}\n            summary[\'sizes\'][size]=item\n            schema=\'\'\n            if any(e.startswith(\'postgres\') for e in args.engines):\n                item[\'postgres_load\']=base.load_postgres(size,folder,params)\n                import psycopg\n                with psycopg.connect(base.PG_DSN,autocommit=True) as con:\n                    item[\'postgres_settings\']={key:con.execute(\'SHOW \'+key).fetchone()[0] for key in [\'server_version\',\'work_mem\',\'shared_buffers\',\'max_parallel_workers_per_gather\']}\n            if \'postgres_index\' in args.engines:\n                schema,item[\'index\']=setup_postgres(f\'{size}_{args.tag}\')\n            (out/\'summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n            references={}\n            for q in args.queries:\n                reference=out/f\'{size}_{q}_reference.parquet\'\n                check=guarded(\'pandas_csv\',q,folder,params[\'customer_key\'],reference,\'\',args)\n                if check[\'status\']!=\'ok\':\n                    raise RuntimeError(f\'Reference failed {q}: {check}\')\n                references[q]=reference\n                print(size,\'reference\',q,\'ok\',flush=True)\n            # Warm every engine equally; warmup failures are recorded and fatal.\n            for engine in args.engines:\n                for q in args.queries:\n                    target=out/f\'{size}_{engine}_{q}.parquet\'\n                    selected=schema if engine==\'postgres_index\' else f\'s_{size}\'\n                    r=guarded(engine,q,folder,params[\'customer_key\'],target,selected,args)\n                    if r[\'status\']!=\'ok\':\n                        raise RuntimeError(f\'Warmup failed {engine}/{q}: {r}\')\n                    compare(references[q],target)\n                    print(size,\'warmup\',engine,q,\'ok\',flush=True)\n                    if engine.startswith(\'postgres\'):\n                        import psycopg\n                        with psycopg.connect(base.PG_DSN,autocommit=True) as con:\n                            con.execute(f\'SET search_path TO {selected}\')\n                            con.execute(f\'SET statement_timeout={int(args.timeout*1000)}\')\n                            plan=con.execute(\'EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) \'+sql(q,params[\'customer_key\'])).fetchone()[0]\n                            (out/f\'{size}_{engine}_{q}_plan.json\').write_text(json.dumps(plan,indent=2),encoding=\'utf-8\')\n            for repeat in range(1,args.repeats+1):\n                jobs=[(e,q) for e in args.engines for q in args.queries]; rng.shuffle(jobs)\n                for engine,q in jobs:\n                    target=out/f\'{size}_{engine}_{q}.parquet\'\n                    r=guarded(engine,q,folder,params[\'customer_key\'],target,schema if engine==\'postgres_index\' else f\'s_{size}\',args)\n                    if r[\'status\']==\'ok\':\n                        try:\n                            compare(references[q],target)\n                            r[\'matches_reference\']=True\n                        except AssertionError as exc:\n                            r.update(status=\'mismatch\',matches_reference=False,error=str(exc)[:1000])\n                    failed |= r[\'status\']!=\'ok\'\n                    r.update(size=size,engine=engine,query=q,repeat=repeat)\n                    item[\'runs\'].append(r); log.write(json.dumps(r)+\'\\n\'); log.flush()\n                    print(size,engine,q,repeat,r[\'status\'],round(r.get(\'seconds\',0),3),flush=True)\n                    (out/\'summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n            item[\'medians\']={}\n            for engine in args.engines:\n                for q in args.queries:\n                    values=[r[\'seconds\'] for r in item[\'runs\'] if r[\'engine\']==engine and r[\'query\']==q and r[\'status\']==\'ok\']\n                    item[\'medians\'][engine+\'/\'+q]=statistics.median(values) if values else None\n            (out/\'summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    with (out/\'comparison.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n        writer=csv.writer(stream)\n        writer.writerow([\'size\',\'query\',\'engine\',\'successful_runs\',\'median_seconds\',\'max_client_peak_gb\',\'speedup_vs_same_engine_baseline\'])\n        for size,item in summary[\'sizes\'].items():\n            for engine in args.engines:\n                for q in args.queries:\n                    runs=[r for r in item[\'runs\'] if r[\'engine\']==engine and r[\'query\']==q and r[\'status\']==\'ok\']\n                    median=item[\'medians\'].get(engine+\'/\'+q)\n                    baseline=\'postgres\' if engine.startswith(\'postgres\') else \'pandas_csv\'\n                    before=item[\'medians\'].get(baseline+\'/\'+q)\n                    writer.writerow([size,q,engine,len(runs),median,max((r[\'client_peak_gb\'] for r in runs),default=None),before/median if before and median else None])\n    print(\'Results:\',out)\n    if failed:\n        sys.exit(1)\n\n\nif __name__==\'__main__\':\n    main()\n', encoding='utf-8')
(WORK / 'src/13_benchmark_postgres_extended.py').write_text('"""Controlled work_mem and materialized-view experiments, measured together."""\nimport argparse\nimport csv\nimport importlib.util\nimport json\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nROOT=Path(__file__).resolve().parents[1]\nspec=importlib.util.spec_from_file_location(\'optimized\',ROOT/\'src/06_benchmark_optimized.py\')\nopt=importlib.util.module_from_spec(spec);spec.loader.exec_module(opt)\nCONFIGS={\'postgres_4mb\':4,\'postgres_32mb\':32,\'postgres_64mb\':64,\'materialized_4mb\':4}\n\ndef query_sql(q,customer,materialized=False):\n    if not materialized:return opt.sql(q,customer)\n    return {\n        \'Q1\':\'SELECT year,month,revenue FROM mv_monthly ORDER BY year,month\',\n        \'Q5\':\'\'\'SELECT d.date,avg(coalesce(m.revenue,0)) OVER\n          (ORDER BY d.date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling\n          FROM dim_date d LEFT JOIN mv_daily m USING(date_key) ORDER BY d.date\'\'\',\n    }[q]\n\ndef configure(con,schema,work_mem,timeout):\n    from psycopg import sql\n    con.execute(sql.SQL(\'SET search_path TO {}\').format(sql.Identifier(schema)))\n    con.execute("SELECT set_config(\'work_mem\',%s,false)",(f\'{work_mem}MB\',))\n    con.execute("SELECT set_config(\'statement_timeout\',%s,false)",(str(int(timeout*1000)),))\n    con.execute(\'SET max_parallel_workers_per_gather=2\')\n\ndef worker(config,q,customer,schema,target,timeout,pidfile):\n    import pandas as pd\n    import psycopg\n    started=time.perf_counter()\n    with psycopg.connect(opt.base.PG_DSN,autocommit=True,application_name=\'books_extended_worker\') as con:\n        Path(pidfile).write_text(str(con.info.backend_pid),encoding=\'ascii\')\n        configure(con,schema,CONFIGS[config],float(timeout))\n        with con.cursor() as cur:\n            cur.execute(query_sql(q,int(customer),config==\'materialized_4mb\'))\n            frame=pd.DataFrame(cur.fetchall(),columns=[c.name for c in cur.description])\n    elapsed=time.perf_counter()-started\n    frame.to_parquet(target,index=False)\n    print(json.dumps({\'seconds\':elapsed,\'rows\':len(frame)}))\n\ndef guarded(config,q,customer,schema,target,args):\n    import psutil\n    pidfile=target.with_suffix(\'.pid\')\n    if pidfile.exists():pidfile.unlink()\n    command=[sys.executable,str(Path(__file__).resolve()),\'--worker\',config,q,str(customer),schema,\n             str(target),str(args.timeout),str(pidfile)]\n    proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,encoding=\'utf-8\')\n    handle=psutil.Process(proc.pid);started=time.perf_counter();peak=0;status=\'ok\'\n    while proc.poll() is None:\n        if psutil.virtual_memory().available<args.min_free_gb*1e9:status=\'low_system_memory\'\n        elif time.perf_counter()-started>args.timeout+30:status=\'timeout\'\n        try:\n            members=[handle]+handle.children(recursive=True)\n            peak=max(peak,sum(m.memory_info().rss for m in members if m.is_running()))\n        except psutil.Error:members=[]\n        if status!=\'ok\':\n            # Cancel only the backend created by this worker before stopping its client.\n            if pidfile.exists():\n                try:\n                    import psycopg\n                    with psycopg.connect(opt.base.PG_DSN,autocommit=True,connect_timeout=5) as con:\n                        con.execute("SELECT pg_cancel_backend(pid) FROM pg_stat_activity WHERE pid=%s AND application_name=\'books_extended_worker\'",(int(pidfile.read_text()),))\n                except Exception:pass\n            for member in reversed(members):\n                try:member.kill()\n                except psutil.Error:pass\n            proc.kill();break\n        time.sleep(.1)\n    out,err=proc.communicate()\n    result={\'status\':status,\'client_peak_gb\':peak/1e9}\n    if status==\'ok\':\n        if proc.returncode:result.update(status=\'error\',error=err[-3000:])\n        else:result.update(json.loads(out.strip().splitlines()[-1]))\n    return result\n\ndef setup(con,schema,size,timeout):\n    from psycopg import sql\n    if con.execute(\'SELECT 1 FROM information_schema.schemata WHERE schema_name=%s\',(schema,)).fetchone():\n        raise RuntimeError(f\'Schema {schema} exists: use a fresh tag.\')\n    indexes=con.execute(\'SELECT indexname FROM pg_indexes WHERE schemaname=%s AND tablename=%s\',(f\'s_{size}\',\'fact_sales\')).fetchall()\n    if indexes:raise RuntimeError(\'Baseline fact already has indexes; use a clean benchmark baseline.\')\n    con.execute(sql.SQL(\'CREATE SCHEMA {}\').format(sql.Identifier(schema)))\n    configure(con,schema,4,timeout)\n    for table in opt.base.TABLES:\n        con.execute(sql.SQL(\'CREATE VIEW {} AS SELECT * FROM {}.{}\').format(sql.Identifier(table),sql.Identifier(\'s_\'+size),sql.Identifier(table)))\n    definitions={\n      \'mv_monthly\':\'\'\'SELECT d.year,d.month,sum(f.estimated_revenue) AS revenue\n        FROM fact_sales f JOIN dim_date d USING(date_key) GROUP BY 1,2\'\'\',\n      \'mv_daily\':\'SELECT date_key,sum(estimated_revenue) AS revenue FROM fact_sales GROUP BY 1\',\n    }\n    info={}\n    for name,query in definitions.items():\n        start=time.perf_counter();con.execute(f\'CREATE MATERIALIZED VIEW {name} AS {query}\')\n        elapsed=time.perf_counter()-start\n        info[name]={\'create_seconds\':elapsed,\'bytes\':con.execute(\'SELECT pg_total_relation_size(%s)\',(f\'{schema}.{name}\',)).fetchone()[0]}\n        start=time.perf_counter();con.execute(f\'ANALYZE {name}\')\n        info[name][\'analyze_seconds\']=time.perf_counter()-start\n        info[name][\'rows\']=con.execute(f\'SELECT count(*) FROM {name}\').fetchone()[0]\n    return info\n\ndef save_summary(out,summary):\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2,default=str),encoding=\'utf-8\')\n\ndef main():\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    if len(sys.argv)>1 and sys.argv[1]==\'--worker\':worker(*sys.argv[2:]);return\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--sizes\',nargs=\'+\',choices=opt.base.SIZES,default=[\'full\'])\n    parser.add_argument(\'--repeats\',type=int,default=5)\n    parser.add_argument(\'--tag\',required=True)\n    parser.add_argument(\'--timeout\',type=float,default=900)\n    parser.add_argument(\'--min-free-gb\',type=float,default=1.0)\n    parser.add_argument(\'--queries\',nargs=\'+\',choices=opt.base.QUERIES,default=opt.base.QUERIES)\n    args=parser.parse_args()\n    if args.repeats<1 or not args.tag.isascii() or not args.tag.replace(\'_\',\'\').isalnum() or len(args.tag)>32:parser.error(\'Invalid tag/repeats\')\n    out=ROOT/\'results\'/args.tag;out.mkdir(exist_ok=False,parents=True)\n    summary={\'machine\':opt.base.machine_info(),\'arguments\':vars(args),\'configs\':CONFIGS,\'sizes\':{},\n             \'timing\':\'Connection + per-session configuration + query + full client materialization; imports, result persistence and comparison excluded. Client RSS excludes PostgreSQL server. Warm OS cache uncontrolled.\'}\n    rng=random.Random(20261110);failed=False;comparison=[]\n    import psycopg\n    with (out/\'runs.jsonl\').open(\'w\',encoding=\'utf-8\') as log:\n        for size in args.sizes:\n            print(\'Preparing:\',size,flush=True)\n            folder=opt.base.prepare(size);params=json.loads((folder/\'params.json\').read_text())\n            item={\'params\':params,\'load\':opt.base.load_postgres(size,folder,params),\'runs\':[],\'warmups\':[]}\n            summary[\'sizes\'][size]=item\n            schema=f\'ex_{size}_{args.tag}\'\n            with psycopg.connect(opt.base.PG_DSN,autocommit=True) as con:\n                item[\'materialized_views\']=setup(con,schema,size,args.timeout)\n                item[\'settings\']={k:con.execute(\'SHOW \'+k).fetchone()[0] for k in [\'server_version\',\'work_mem\',\'hash_mem_multiplier\',\'shared_buffers\',\'max_parallel_workers_per_gather\',\'block_size\']}\n            save_summary(out,summary)\n            references={}\n            for q in args.queries:\n                target=out/f\'{size}_{q}_reference.parquet\'\n                r=guarded(\'postgres_4mb\',q,params[\'customer_key\'],schema,target,args)\n                if r[\'status\']!=\'ok\':raise RuntimeError(f\'Reference {q} failed: {r}\')\n                references[q]=target;print(size,\'reference\',q,\'ok\',flush=True)\n            jobs=[(config,q) for config in CONFIGS for q in args.queries if config!=\'materialized_4mb\' or q in [\'Q1\',\'Q5\']]\n            for config,q in jobs:\n                target=out/f\'{size}_{config}_{q}.parquet\'\n                r=guarded(config,q,params[\'customer_key\'],schema,target,args)\n                r.update(config=config,query=q)\n                item[\'warmups\'].append(r);save_summary(out,summary)\n                if r[\'status\']!=\'ok\':raise RuntimeError(f\'Warmup failed: {r}\')\n                opt.compare(references[q],target)\n                print(size,\'warmup\',config,q,\'ok\',flush=True)\n                with psycopg.connect(opt.base.PG_DSN,autocommit=True) as con:\n                    configure(con,schema,CONFIGS[config],args.timeout)\n                    plan=con.execute(\'EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) \'+query_sql(q,params[\'customer_key\'],config==\'materialized_4mb\')).fetchone()[0]\n                    (out/f\'{size}_{config}_{q}_plan.json\').write_text(json.dumps(plan,indent=2),encoding=\'utf-8\')\n            for repeat in range(1,args.repeats+1):\n                plan=list(jobs);rng.shuffle(plan)\n                for config,q in plan:\n                    target=out/f\'{size}_{config}_{q}.parquet\'\n                    r=guarded(config,q,params[\'customer_key\'],schema,target,args)\n                    if r[\'status\']==\'ok\':\n                        try:opt.compare(references[q],target);r[\'matches_reference\']=True\n                        except AssertionError as exc:r.update(status=\'mismatch\',matches_reference=False,error=str(exc)[:2000])\n                    r.update(size=size,config=config,query=q,repeat=repeat)\n                    item[\'runs\'].append(r);log.write(json.dumps(r)+\'\\n\');log.flush()\n                    failed|=r[\'status\']!=\'ok\';save_summary(out,summary)\n                    print(size,config,q,repeat,r[\'status\'],round(r.get(\'seconds\',0),3),flush=True)\n            # Full, nonconcurrent refresh with no source mutation: measures recomputation,\n            # not incremental maintenance, ingestion, or freshness under writes.\n            with psycopg.connect(opt.base.PG_DSN,autocommit=True) as con:\n                configure(con,schema,4,args.timeout)\n                for name,info in item[\'materialized_views\'].items():\n                    values=[]\n                    for _ in range(args.repeats):\n                        start=time.perf_counter();con.execute(f\'REFRESH MATERIALIZED VIEW {name}\');values.append(time.perf_counter()-start)\n                    info[\'refresh_seconds\']=values;info[\'refresh_median_s\']=statistics.median(values)\n                    q=\'Q1\' if name==\'mv_monthly\' else \'Q5\'\n                    with con.cursor() as cur:\n                        import pandas as pd\n                        cur.execute(query_sql(q,params[\'customer_key\'],True))\n                        frame=pd.DataFrame(cur.fetchall(),columns=[c.name for c in cur.description])\n                    target=out/f\'{size}_{name}_after_refresh.parquet\';frame.to_parquet(target,index=False)\n                    # Check even when Q1/Q5 were not among measured queries.\n                    if q not in references:\n                        reference=out/f\'{size}_{q}_refresh_reference.parquet\'\n                        checked=guarded(\'postgres_4mb\',q,params[\'customer_key\'],schema,reference,args)\n                        if checked[\'status\']!=\'ok\':raise RuntimeError(checked)\n                    else:reference=references[q]\n                    opt.compare(reference,target);info[\'refresh_matches_reference\']=True\n                    print(size,\'refresh\',name,\'ok\',flush=True)\n            for config,q in jobs:\n                rr=[r for r in item[\'runs\'] if r[\'config\']==config and r[\'query\']==q and r[\'status\']==\'ok\']\n                baseline=[r[\'seconds\'] for r in item[\'runs\'] if r[\'config\']==\'postgres_4mb\' and r[\'query\']==q and r[\'status\']==\'ok\']\n                med=statistics.median(r[\'seconds\'] for r in rr) if rr else None\n                row={\'size\':size,\'config\':config,\'query\':q,\'successful_runs\':len(rr),\'median_s\':med,\n                     \'min_s\':min((r[\'seconds\'] for r in rr),default=None),\'max_s\':max((r[\'seconds\'] for r in rr),default=None),\n                     \'speedup_vs_4mb\':statistics.median(baseline)/med if baseline and med else None}\n                if config==\'materialized_4mb\' and med and baseline:\n                    name=\'mv_monthly\' if q==\'Q1\' else \'mv_daily\'\n                    saving=statistics.median(baseline)-med\n                    row[\'create_break_even_queries\']=__import__(\'math\').ceil(item[\'materialized_views\'][name][\'create_seconds\']/saving) if saving>0 else None\n                comparison.append(row)\n            save_summary(out,summary)\n            with (out/\'comparison.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n                keys=[\'size\',\'config\',\'query\',\'successful_runs\',\'median_s\',\'min_s\',\'max_s\',\'speedup_vs_4mb\',\'create_break_even_queries\']\n                writer=csv.DictWriter(f,fieldnames=keys);writer.writeheader();writer.writerows(comparison)\n    print(\'COMPLETE:\',out,flush=True)\n    if failed:raise SystemExit(1)\n\nif __name__==\'__main__\':main()\n', encoding='utf-8')
(WORK / 'src/test_benchmark_optimized.py').write_text('"""Correctness on edge cases that aggregate fingerprints can miss."""\nimport importlib.util\nimport tempfile\nimport unittest\nfrom pathlib import Path\n\nspec = importlib.util.spec_from_file_location(\'optimized\', Path(__file__).with_name(\'06_benchmark_optimized.py\'))\nopt = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(opt)\n\n\nclass QueryTests(unittest.TestCase):\n    def test_collation_order_does_not_change_row_equality(self):\n        import pandas as pd\n        with tempfile.TemporaryDirectory() as tmp:\n            left, right = Path(tmp) / \'left.parquet\', Path(tmp) / \'right.parquet\'\n            rows = pd.DataFrame({\'category\': [\'#Booktok\', \'ASINs for HQP\', \'Arts & Photography\', None],\n                                 \'book_key\': [1, 2, 3, 4], \'n\': [2, 3, 4, 5]})\n            rows.to_parquet(left, index=False)\n            rows.iloc[[2, 1, 3, 0]].to_parquet(right, index=False)\n            opt.compare(left, right)\n            rows.loc[1, \'book_key\'] = 99\n            rows.to_parquet(right, index=False)\n            with self.assertRaises(AssertionError):\n                opt.compare(left, right)\n\n    def test_cross_engine_nulls_ties_dates_and_rfm(self):\n        import pandas as pd\n        import duckdb\n        with tempfile.TemporaryDirectory() as tmp:\n            folder = Path(tmp)\n            data = {\n                \'fact_sales\': pd.DataFrame([\n                    [1, 20230101, 1, 1, 1, 10., 10., 5., 0],\n                    [2, 20230101, 1, 2, 1, None, None, 4., 0],\n                    [3, 20230103, 2, 3, 1, None, None, 3., 0],\n                    [4, 20230201, 3, 3, 1, None, None, 3., 0],\n                ], columns=[p.strip().split()[0] for p in opt.base.DDL[\'fact_sales\'].split(\',\')]),\n                \'dim_books\': pd.DataFrame([\n                    [1, \'a\', \'A\', \'Author\', \'Fiction\', \'\', \'001\', 2020, \'en\', 100, 10., 5., 1],\n                    [2, \'b\', \'B\', \'Author\', \'Fiction\', \'\', \'002\', 2020, \'en\', 100, None, 4., 1],\n                    [3, \'c\', \'C\', \'Author\', None, \'\', \'003\', 2020, \'en\', 100, None, 3., 2],\n                ], columns=[p.strip().split()[0] for p in opt.base.DDL[\'dim_books\'].split(\',\')]),\n                \'dim_date\': pd.DataFrame([\n                    [20230101, \'2023-01-01\', 1, 1, 1, 1, 2023, True],\n                    [20230102, \'2023-01-02\', 2, 1, 1, 1, 2023, False],\n                    [20230103, \'2023-01-03\', 3, 1, 1, 1, 2023, False],\n                    [20230201, \'2023-02-01\', 1, 5, 2, 1, 2023, False],\n                ], columns=[p.strip().split()[0] for p in opt.base.DDL[\'dim_date\'].split(\',\')]),\n            }\n            for table, df in data.items():\n                df.to_csv(folder / f\'{table}.csv\', index=False)\n                df.to_parquet(folder / f\'{table}.parquet\', index=False)\n            with duckdb.connect() as con:\n                for table in data:\n                    con.execute(f"CREATE VIEW {table} AS SELECT * FROM read_csv(\'{folder.as_posix()}/{table}.csv\',header=true)")\n                for q in opt.base.QUERIES:\n                    reference = folder / \'reference.parquet\'\n                    result = opt.pandas_query(\'pandas_csv\', q, folder, 1)\n                    result.to_parquet(reference, index=False)\n                    for engine in [\'pandas_columns\', \'pandas_parquet\']:\n                        actual = folder / \'actual.parquet\'\n                        opt.pandas_query(engine, q, folder, 1).to_parquet(actual, index=False)\n                        opt.compare(reference, actual)\n                    actual = folder / \'sql.parquet\'\n                    con.execute(opt.sql(q, 1)).fetchdf().to_parquet(actual, index=False)\n                    opt.compare(reference, actual)\n            q1 = opt.pandas_query(\'pandas_csv\', \'Q1\', folder, 1)\n            self.assertTrue(pd.isna(q1.loc[q1.month == 2, \'revenue\'].iloc[0]))\n            q6 = opt.pandas_query(\'pandas_csv\', \'Q6\', folder, 1)\n            self.assertEqual(q6.loc[q6.customer_key == 2, \'monetary\'].iloc[0], 0)\n            # An empty customer history must also have matching output columns.\n            a = opt.pandas_query(\'pandas_csv\', \'Q3\', folder, 999)\n            self.assertEqual(len(a), 0)\n\n    def test_row_comparison_detects_same_sum_different_rows(self):\n        import pandas as pd\n        with tempfile.TemporaryDirectory() as tmp:\n            left, right = Path(tmp) / \'left.parquet\', Path(tmp) / \'right.parquet\'\n            pd.DataFrame({\'value\': [1, 4]}).to_parquet(left)\n            pd.DataFrame({\'value\': [2, 3]}).to_parquet(right)\n            with self.assertRaises(AssertionError):\n                opt.compare(left, right)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', encoding='utf-8')
(WORK / 'src/test_postgres_extended.py').write_text('"""Integration test: NULL/month boundaries, work_mem sessions and MV refresh."""\nimport importlib.util\nimport tempfile\nimport unittest\nimport uuid\nfrom pathlib import Path\nimport pandas as pd\nimport psycopg\n\nspec=importlib.util.spec_from_file_location(\'extended\',Path(__file__).with_name(\'13_benchmark_postgres_extended.py\'))\next=importlib.util.module_from_spec(spec);spec.loader.exec_module(ext)\n\nclass MaterializedViewTests(unittest.TestCase):\n    def test_nulls_gaps_and_refresh(self):\n        token=uuid.uuid4().hex[:10]\n        size=\'fixture_\'+token;baseline=\'s_\'+size;schema=\'test_mv_\'+token\n        with psycopg.connect(ext.opt.base.PG_DSN,autocommit=True) as con:\n            try:\n                con.execute(f\'CREATE SCHEMA {baseline}\')\n                con.execute(f\'CREATE TABLE {baseline}.fact_sales(date_key INT,estimated_revenue DOUBLE PRECISION)\')\n                con.execute(f\'CREATE TABLE {baseline}.dim_date(date_key INT,date DATE,year INT,month INT)\')\n                con.execute(f\'CREATE TABLE {baseline}.dim_books(book_key INT)\')\n                con.execute(f\'CREATE TABLE {baseline}.dim_customers(customer_key INT)\')\n                con.execute(f"INSERT INTO {baseline}.dim_date VALUES (20230101,\'2023-01-01\',2023,1),(20230102,\'2023-01-02\',2023,1),(20230103,\'2023-01-03\',2023,1),(20230201,\'2023-02-01\',2023,2)")\n                con.execute(f\'INSERT INTO {baseline}.fact_sales VALUES (20230101,10),(20230101,20),(20230102,NULL),(20230201,NULL)\')\n                info=ext.setup(con,schema,size,30)\n                self.assertEqual(info[\'mv_monthly\'][\'rows\'],2)\n                self.assertEqual(info[\'mv_daily\'][\'rows\'],3)\n                def frame(q,mv):\n                    with con.cursor() as cur:\n                        cur.execute(ext.query_sql(q,1,mv))\n                        return pd.DataFrame(cur.fetchall(),columns=[c.name for c in cur.description])\n                with tempfile.TemporaryDirectory() as tmp:\n                    a,b=Path(tmp)/\'a.parquet\',Path(tmp)/\'b.parquet\'\n                    for q in [\'Q1\',\'Q5\']:\n                        frame(q,False).to_parquet(a,index=False);frame(q,True).to_parquet(b,index=False)\n                        ext.opt.compare(a,b)\n                    self.assertTrue(pd.isna(frame(\'Q1\',True).iloc[1].revenue))\n                    # A new source row does not silently update an MV.\n                    con.execute(f\'INSERT INTO {baseline}.fact_sales VALUES (20230103,7)\')\n                    self.assertEqual(frame(\'Q1\',True).iloc[0].revenue,30)\n                    for name in [\'mv_monthly\',\'mv_daily\']:con.execute(f\'REFRESH MATERIALIZED VIEW {name}\')\n                    self.assertEqual(frame(\'Q1\',True).iloc[0].revenue,37)\n                    for q in [\'Q1\',\'Q5\']:\n                        frame(q,False).to_parquet(a,index=False);frame(q,True).to_parquet(b,index=False)\n                        ext.opt.compare(a,b)\n                ext.configure(con,schema,64,30)\n                self.assertEqual(con.execute(\'SHOW work_mem\').fetchone()[0],\'64MB\')\n                ext.configure(con,schema,4,30)\n                self.assertEqual(con.execute(\'SHOW work_mem\').fetchone()[0],\'4MB\')\n            finally:\n                # Only the two randomly named schemas created by this test are removed.\n                con.execute(f\'DROP SCHEMA IF EXISTS {schema} CASCADE\')\n                con.execute(f\'DROP SCHEMA IF EXISTS {baseline} CASCADE\')\n\nif __name__==\'__main__\':unittest.main()\n', encoding='utf-8')


In [ ]:
# 3. Cài thư viện và PostgreSQL với cấu hình mặc định, cổng 55432
pip = subprocess.run(f'{sys.executable} -m pip install -q "duckdb==2.0.0" "numpy==2.4.6" "pandas==3.1.1" "psutil==7.2.2" "psycopg==3.3.8" "psycopg-binary==3.3.8" "pyarrow==26.0.1"', shell=True, capture_output=True, text=True)
print(pip.stdout[-800:], pip.stderr[-800:])
assert pip.returncode == 0, 'Cài thư viện thất bại.'

setup = r'''
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get -qq update
apt-get -qq -y install postgresql > /dev/null
CONF=$(ls -d /etc/postgresql/*/main | head -1)
service postgresql stop || true
sed -i -E "s/^#?port = .*/port = 55432/" "$CONF/postgresql.conf"
sed -i -E "s/^#?listen_addresses = .*/listen_addresses = '127.0.0.1'/" "$CONF/postgresql.conf"
sed -i -E "s/^(host[[:space:]]+all[[:space:]]+all[[:space:]]+127\.0\.0\.1\/32[[:space:]]+).*/\1trust/" "$CONF/pg_hba.conf"
service postgresql start
sleep 4
su postgres -c "psql -p 55432 -tAc \"SELECT 1 FROM pg_database WHERE datname='book_sales'\"" | grep -q 1 || su postgres -c "createdb -p 55432 book_sales"
su postgres -c "psql -p 55432 -tAc 'select version()'"
'''
result = subprocess.run(['bash', '-c', setup], capture_output=True, text=True)
print(result.stdout[-800:], result.stderr[-800:])
assert result.returncode == 0, 'Cài PostgreSQL thất bại.'

check = subprocess.run([sys.executable, '-c', "import psycopg; print(psycopg.connect('host=127.0.0.1 port=55432 user=postgres dbname=book_sales').execute('select 1').fetchone())"], capture_output=True, text=True)
print(check.stdout, check.stderr[-500:])
assert check.returncode == 0, 'Không kết nối được PostgreSQL.'
versions = subprocess.run(f'{sys.executable} -m pip freeze', shell=True, capture_output=True, text=True).stdout
(SESSION_DIR / 'environment.json').write_text(json.dumps({
    'python': platform.python_version(), 'ram_gb': round(ram_gb, 1), 'cpu': cpu[0], 'cpu_count': len(cpu),
    'postgres': result.stdout.strip().splitlines()[-1], 'pip_freeze': versions.splitlines()}, indent=2, ensure_ascii=False))
print('Môi trường sẵn sàng')
# Run the small correctness tests before the expensive experiment.
tested = subprocess.run([sys.executable, str(WORK / 'src/test_benchmark_optimized.py')],
                        cwd=WORK, capture_output=True, text=True)
print(tested.stdout, tested.stderr)
assert tested.returncode == 0, 'Correctness tests failed; benchmark not started.'
import pyarrow.parquet as pq
rows = pq.ParquetFile(WORK / 'data/warehouse/fact_sales.parquet').metadata.num_rows
assert rows == 20_342_188, f'Unexpected full dataset size: {rows:,}'
print('Full dataset:', f'{rows:,}', 'rows')

# Validate MV semantics and per-session work_mem in isolated temporary schemas.
tested = subprocess.run([sys.executable, str(WORK / 'src/test_postgres_extended.py')],
                        cwd=WORK, capture_output=True, text=True)
print(tested.stdout, tested.stderr)
assert tested.returncode == 0, 'Extended PostgreSQL tests failed.'


In [ ]:
# 4. Measure new baseline and extended configurations in this runtime.
completed = []
for size in PLAN:
    tag = 'pgx_' + size + '_' + SESSION
    local_results = WORK / 'results' / tag
    saved_results = SESSION_DIR / tag
    saved_results.mkdir(exist_ok=False)
    command = [sys.executable, '-u', str(WORK / 'src/13_benchmark_postgres_extended.py'),
               '--sizes', size, '--repeats', str(REPEATS), '--tag', tag,
               '--min-free-gb', '1.0', '--timeout', '900']
    print('START:', size, '|', saved_results, flush=True)
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with (saved_results / 'console.log').open('w', encoding='utf-8') as console:
            for line in process.stdout:
                print(line, end='', flush=True)
                console.write(line); console.flush()
                for name in ['summary.json', 'runs.jsonl']:
                    if (local_results / name).exists():
                        shutil.copy2(local_results / name, saved_results / name)
            process.wait()
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
        if local_results.exists():
            shutil.copytree(local_results, saved_results, dirs_exist_ok=True)
    assert process.returncode == 0, f'Incomplete {size}; inspect {saved_results}/console.log'
    summary = json.loads((saved_results / 'summary.json').read_text())
    item = summary['sizes'][size]
    runs = item['runs']
    assert len(runs) == 20 * REPEATS
    assert all(r['status'] == 'ok' and r.get('matches_reference') for r in runs)
    assert all(v['refresh_matches_reference'] for v in item['materialized_views'].values())
    completed.append({'size': size, 'tag': tag, 'measured_runs': len(runs)})
    (SESSION_DIR / 'completed.json').write_text(json.dumps(completed, indent=2))
    print('SAVED:', size, '|', len(runs), 'verified measurements', flush=True)
print('EXTENDED BENCHMARK COMPLETE:', SESSION_DIR)


In [ ]:
# 5. Export query medians plus separate preparation/refresh costs.
import pandas as pd
tables = []
costs = []
for record in completed:
    path = SESSION_DIR / record['tag']
    tables.append(pd.read_csv(path / 'comparison.csv'))
    info = json.loads((path / 'summary.json').read_text())['sizes'][record['size']]
    for name, values in info['materialized_views'].items():
        costs.append({'size': record['size'], 'view': name,
                      'create_seconds': values['create_seconds'],
                      'analyze_seconds': values['analyze_seconds'],
                      'refresh_median_s': values['refresh_median_s'],
                      'bytes': values['bytes'], 'rows': values['rows']})
comparison = pd.concat(tables, ignore_index=True)
comparison.to_csv(SESSION_DIR / 'comparison_extended_all_sizes.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(costs).to_csv(SESSION_DIR / 'materialized_view_costs.csv', index=False, encoding='utf-8-sig')
display(comparison[comparison['size']=='full'])
display(pd.DataFrame(costs))
print('Download results folder after completion:', SESSION_DIR)


## Diễn giải đúng

- work_mem là giới hạn theo thao tác/worker, không phải tổng RAM server. Hash còn phụ thuộc hash_mem_multiplier.
- REFRESH ở đây là tính lại toàn bộ trên nguồn không thay đổi, không phải cập nhật gia tăng hay ghi đồng thời.
- View cần làm mới khi nguồn đổi; tốc độ truy vấn không miễn phí, phải báo cả thời gian tạo/refresh/dung lượng.
- Không trộn số liệu mới với benchmark cũ để tính speedup; dùng postgres_4mb cùng phiên này.
- Mốc hòa vốn chỉ tính từ chi phí tạo và thời gian tiết kiệm mỗi truy vấn, chưa tính chi phí làm mới.
- Kết quả máy cá nhân chỉ để kiểm tra; chưa đưa vào báo cáo như kết quả Colab.
